### Regex(Regular Expression)
#### 1. Character Classes (The "What")
Character classes tell the computer what kind of character it is looking for.

Instead of typing exact letters or numbers, you use these shortcuts:

* \d (Digit): Matches any single number from 0 to 9.

Example: \d in "Agent 007" will match the 0, the 0, and the 7 individually.

* \D (Non-Digit): The exact opposite. It matches anything that is NOT a number (letters, spaces, punctuation).

Example: \D in "ID: 45" matches "I", "D", ":", and the space.

* \w (Word): Matches any standard letter, number, or underscore. It does not match spaces or special symbols like @ or !.

* \W (Non-Word): Matches only spaces and special symbols. (Very useful for finding and deleting junk characters in dirty data).

* [abc] (Custom Set): You make your own rules. This tells Python to match only a, b, or c.

Example: [aeiou] will find only the vowels in a word.

* [^abc] (Negated Set): Adding the ^ symbol inside the brackets means "NOT". It matches anything except a, b, or c.

#### 2. Quantifiers (The "How Many")
By default, Character Classes (like \d) only look for one single character at a time. Quantifiers sit right next to a character class and tell Python how many times in a row to look for it.

* + (One or more): This is the most used quantifier. It groups consecutive matches together.

Without +: \d looking at "Price: 500" finds 5, then 0, then 0.

With +: \d+ looking at "Price: 500" finds the entire number "500" as one single unit.

* ? (Zero or one - Optional): Makes the preceding character optional.

Example: colou?r means the u is optional. It will successfully match both "color" and "colour".

* {n} (Exactly n times): Looks for a specific length.

Example: \d{4} means find exactly 4 numbers in a row (perfect for finding a 4-digit PIN like "1234").

*{n,m} (Between n and m times): Looks for a range.

Example: \d{2,4} will match numbers that are 2, 3, or 4 digits long.

* '*' (Zero or more): Matches everything or nothing. Often used as a wildcard to skip over unpredictable text.

#### 3. Groups (The "Package")
Groups are created using parentheses (). They do two very important things:

* 1. Treating multiple characters as one unit:
If you want to look for the exact sequence "ha" repeating, you group it: (ha)+. This will match "hahaha". (If you just wrote ha+, it would match "haaaa").

* 2. Extracting specific data (Capturing):
This is the most powerful feature for a Data Engineer. It allows you to match a whole pattern to ensure the data is valid, but only "capture" the piece you actually care about.

##### Scenario: You have a text string "Date: 15-08-2026". You only want to extract the year.

* The Pattern: You write the regex for the whole date so Python knows exactly what it's looking at: \d{2}-\d{2}-\d{4}.

* The Group: You put parentheses only around the year part: \d{2}-\d{2}-(\d{4}).

* The Result: Python finds the full date, but only pulls out "2026" for you to save in your database.

##### Putting It All Together (A Quick Walkthrough)
Imagine you have this messy string:
"Contact the support team at 987-654-3210 immediately."

* You want to find the phone number.

* You know it starts with 3 digits: \d{3}

* Then a dash: -

* Then 3 more digits: \d{3}

* Then a dash: -

* Then 4 digits: \d{4}

* Your final Regex pattern is simply: \d{3}-\d{3}-\d{4}.

#### 1. Character Classes

##### The first 10 questions focused strictly on Character Classes (\d, \D, \w, \W, [abc], [^abc]).
##### Practice Questions (Part 1: Character Classes)
For these exercises, use Pandas functions like .str.findall(), .str.replace(), or .str.contains(). Because we are not using quantifiers (like + or *) yet, expect functions like findall to return individual characters in a list (e.g., ['1', '9', '2']).

In [2]:
import pandas as pd
import io

csv_data = """Log_ID,IP_Address,User_Account,Event_Status,Raw_Log_Message
L-001,192.168.1.15,admin_root,FAILED!,Attempt #1: Login failed @ 23:59.
L-002,10.0.0.45,guest01,SUCCESS,User guest01 logged in. (Port: 443)
L-003,172.16.254.1,sys-admin,ERROR_500,CRITICAL: DB connection lost! Retrying...
L-004,192.168.1.200,dev_test,BLOCKED,Malware payload caught in quarantine.
L-005,10.0.0.99,unknown,DENIED,Access denied. Invalid password "$uperSecr3t*".
"""

df = pd.read_csv(io.StringIO(csv_data))
df

,Log_ID,IP_Address,User_Account,Event_Status,Raw_Log_Message
0,L-001,192.168.1.15,admin_root,FAILED!,Attempt #1: Login failed @ 23:59.
1,L-002,10.0.0.45,guest01,SUCCESS,User guest01 logged in. (Port: 443)
2,L-003,172.16.254.1,sys-admin,ERROR_500,CRITICAL: DB connection lost! Retrying...
3,L-004,192.168.1.200,dev_test,BLOCKED,Malware payload caught in quarantine.
4,L-005,10.0.0.99,unknown,DENIED,"Access denied. Invalid password ""$uperSecr3t*""."


##### 1. Find Digits: 
Use \d and .str.findall() to extract every single individual number from the Raw_Log_Message column.

In [3]:
df['Raw_Log_Message'].str.findall(r'\d')

0    [1, 2, 3, 5, 9]
1    [0, 1, 4, 4, 3]
2                 []
3                 []
4                [3]
Name: Raw_Log_Message, dtype: object

##### 2. Remove Non-Digits: 
Use \D and .str.replace() to replace everything that is not a number in the Log_ID column with an empty string "".

In [4]:
df['Log_ID'].str.replace(r'\D',"")

0    L-001
1    L-002
2    L-003
3    L-004
4    L-005
Name: Log_ID, dtype: str

##### 3. Find Word Characters: 
Use \w and .str.findall() to extract all valid word characters (letters, numbers, underscores) from the Event_Status column.

In [5]:
df['Event_Status'].str.findall(r'\w')

0             [F, A, I, L, E, D]
1          [S, U, C, C, E, S, S]
2    [E, R, R, O, R, _, 5, 0, 0]
3          [B, L, O, C, K, E, D]
4             [D, E, N, I, E, D]
Name: Event_Status, dtype: object

##### 4. Remove Special Characters: 
Use \W and .str.replace() to replace all non-word characters in the User_Account column (like the hyphen in sys-admin) with an underscore _.

In [6]:
df['User_Account'].str.findall(r'\W')

0     []
1     []
2    [-]
3     []
4     []
Name: User_Account, dtype: object

##### 5. Custom Set (Vowels): 
Use [aeiou] and .str.findall() to find all the lowercase vowels used in the User_Account column.

In [7]:
df['User_Account'].str.findall(r'[aeiou]')

0    [a, i, o, o]
1          [u, e]
2          [a, i]
3          [e, e]
4          [u, o]
Name: User_Account, dtype: object

##### 6. Negated Set (Not Vowels): 
Use [^aeiou] and .str.findall() on the User_Account column to extract everything that is not a lowercase vowel.

In [8]:
df['User_Account'].str.findall(r'[^aeiou]')

0       [d, m, n, _, r, t]
1          [g, s, t, 0, 1]
2    [s, y, s, -, d, m, n]
3       [d, v, _, t, s, t]
4          [n, k, n, w, n]
Name: User_Account, dtype: object

##### 7. Custom Set (Uppercase): 
Use [A-Z] and .str.findall() to extract all uppercase letters from the Raw_Log_Message column.

In [9]:
df['Raw_Log_Message'].str.findall(r'[A-Z]')

0                               [A, L]
1                               [U, P]
2    [C, R, I, T, I, C, A, L, D, B, R]
3                                  [M]
4                            [A, I, S]
Name: Raw_Log_Message, dtype: object

##### 8. Custom Set (Numbers): 
Use [0-9] and .str.contains() to filter the DataFrame and print only the rows where the User_Account contains a number.

In [10]:
df['User_Account'].str.contains(r'[0-9]')

0    False
1     True
2    False
3    False
4    False
Name: User_Account, dtype: bool

##### 9. Find Specific Punctuation: 
Use a custom set like [!@#$*] and .str.findall() to extract only those specific special characters from Raw_Log_Message.

In [12]:
df['Raw_Log_Message'].str.findall(r'[!@#$*]')

0    [#, @]
1        []
2       [!]
3        []
4    [$, *]
Name: Raw_Log_Message, dtype: object

##### 10. Targeted Deletion: 
Use [a-z] and .str.replace() to replace all lowercase letters in the Event_Status column with an empty string "".

In [13]:
df['Event_Status'].str.replace(r'[a-z]',"")

0      FAILED!
1      SUCCESS
2    ERROR_500
3      BLOCKED
4       DENIED
Name: Event_Status, dtype: str

#### Brand new dataset and 8 questions strictly using Character Classes (\d, \D, \w, \W, [abc], [^abc]). No quantifiers, no groups, and no hints.

In [14]:
import pandas as pd
import io

csv_data = """Txn_ID,Client_Ref,Server_Host,Status_Code,System_Message
T-901,USR_881,192.168.0.1,ERR_502,Gateway timeout! Retry @ 14:00.
T-902,USR_B2C,10.0.0.55,OK_200,Payload delivered successfully.
T-903,SYS_ADM,172.16.0.2,ERR_403,Access Denied: Invalid key '$ecr3t'.
T-904,USR_999,192.168.0.100,WARN_99,Disk space low... 89% full.
"""

df = pd.read_csv(io.StringIO(csv_data))
df

,Txn_ID,Client_Ref,Server_Host,Status_Code,System_Message
0,T-901,USR_881,192.168.0.1,ERR_502,Gateway timeout! Retry @ 14:00.
1,T-902,USR_B2C,10.0.0.55,OK_200,Payload delivered successfully.
2,T-903,SYS_ADM,172.16.0.2,ERR_403,Access Denied: Invalid key '$ecr3t'.
3,T-904,USR_999,192.168.0.100,WARN_99,Disk space low... 89% full.


1. Extract every individual digit from the Status_Code column.

In [15]:
df['Status_Code'].str.findall(r'\d')

0    [5, 0, 2]
1    [2, 0, 0]
2    [4, 0, 3]
3       [9, 9]
Name: Status_Code, dtype: object

2. Replace every number in the Txn_ID column with an empty string "".

In [17]:
df['Txn_ID'].str.replace(r'\d',"")

0    T-901
1    T-902
2    T-903
3    T-904
Name: Txn_ID, dtype: str

3. Extract every individual non-word character from the System_Message column.

In [18]:
df['System_Message'].str.findall(r'\W')

0       [ , !,  ,  , @,  , :, .]
1                      [ ,  , .]
2    [ , :,  ,  ,  , ', $, ', .]
3    [ ,  , ., ., .,  , %,  , .]
Name: System_Message, dtype: object

4. Filter the DataFrame to return only the rows where the Client_Ref column contains a number.

In [19]:
df['Client_Ref'].str.contains(r'\d')

0     True
1     True
2    False
3     True
Name: Client_Ref, dtype: bool

5. Extract all lowercase vowels from the System_Message column.

In [20]:
df['System_Message'].str.findall(r'[a-z]')

0    [a, t, e, w, a, y, t, i, m, e, o, u, t, e, t, ...
1    [a, y, l, o, a, d, d, e, l, i, v, e, r, e, d, ...
2    [c, c, e, s, s, e, n, i, e, d, n, v, a, l, i, ...
3        [i, s, k, s, p, a, c, e, l, o, w, f, u, l, l]
Name: System_Message, dtype: object

6. Extract every individual character that is not a number from the Client_Ref column.

In [22]:
df['Client_Ref'].str.findall(r'\D')

0             [U, S, R, _]
1       [U, S, R, _, B, C]
2    [S, Y, S, _, A, D, M]
3             [U, S, R, _]
Name: Client_Ref, dtype: object

7. Filter the DataFrame to return only the rows where the Status_Code column contains either the uppercase letter "E" or the uppercase letter "W".

In [26]:
df['Status_Code'].str.contains(r'[EW]')

0     True
1    False
2     True
3     True
Name: Status_Code, dtype: bool

8. Replace specifically the exclamation mark "!", the period ".", and the apostrophe "'" in the System_Message column with an empty string "".

In [30]:
df['System_Message'].str.replace(r"[!.']","",regex = True)

0        Gateway timeout Retry @ 14:00
1       Payload delivered successfully
2    Access Denied: Invalid key $ecr3t
3              Disk space low 89% full
Name: System_Message, dtype: str

df

#### 2. Quantifiers

##### Quantifiers (+, ?, *, {n}, {n,m}). These symbols allow you to pull out entire words, complete phone numbers, and variable-length IDs instead of just single characters.

In [32]:
import pandas as pd
import io

csv_data = """Event_ID,User_Email,Phone_Number,Transaction_Ref,Error_Message
E-1001,alice.smith@gmail.com,9876543210,TXN-9999,Payment timeout after 30s.
E-1002,bob_jones_89@yahoo.com,+91-9988776655,TX-88,Card declined! Error code: 5021.
E-1003,charlie.d@zoho.in,8877665544,TXN-7,User dropped from session.
E-1004,dave-99@company.com,044-2345678,TXNN-55555,Retrying connection... attempts: 3.
E-1005,eve_test@domain.org,987-654-3210,T-1234,SUCCESS.
"""

df = pd.read_csv(io.StringIO(csv_data))
df

,Event_ID,User_Email,Phone_Number,Transaction_Ref,Error_Message
0,E-1001,alice.smith@gmail.com,9876543210,TXN-9999,Payment timeout after 30s.
1,E-1002,bob_jones_89@yahoo.com,+91-9988776655,TX-88,Card declined! Error code: 5021.
2,E-1003,charlie.d@zoho.in,8877665544,TXN-7,User dropped from session.
3,E-1004,dave-99@company.com,044-2345678,TXNN-55555,Retrying connection... attempts: 3.
4,E-1005,eve_test@domain.org,987-654-3210,T-1234,SUCCESS.


##### Question 1: One or More (+)
Extract all full, multi-digit numbers from the Error_Message column.

Hint: Use \d combined with the + quantifier inside .str.findall().

In [33]:
df['Error_Message'].str.findall(r'\d+')

0      [30]
1    [5021]
2        []
3       [3]
4        []
Name: Error_Message, dtype: object

##### Question 2: Exactly n times ({n})
Filter the DataFrame to show only rows where the Phone_Number column contains exactly 10 consecutive digits without any hyphens or symbols.

Hint: Use \d{10} inside .str.contains().

In [36]:
df['Phone_Number'].str.contains(r'\d{10}')

0     True
1     True
2     True
3    False
4    False
Name: Phone_Number, dtype: bool

##### Question 3: Exactly n times ({n})
Extract only the 4-digit error codes or numbers from the Error_Message column.

Hint: Use \d{4} inside .str.findall().

In [37]:
df['Error_Message'].str.findall(r'\d{4}')

0        []
1    [5021]
2        []
3        []
4        []
Name: Error_Message, dtype: object

##### Question 4: Range ({n,m})
Extract the numeric parts of the Transaction_Ref column that are strictly between 1 and 2 digits long.

Hint: Use \d{1,2} inside .str.findall().

In [38]:
df['Transaction_Ref'].str.findall(r'\d{1,2}')

0       [99, 99]
1           [88]
2            [7]
3    [55, 55, 5]
4       [12, 34]
Name: Transaction_Ref, dtype: object

##### Question 5: Optional Character (?)
The transaction prefix varies (TXN-, TX-, or T-). Extract the prefix starting with "T", followed by an optional "X", an optional "N", and a hyphen.

Hint: Use TX?N?- inside .str.findall() on the Transaction_Ref column.

In [39]:
df['Transaction_Ref'].str.findall(r'TX?N?')

0    [TXN]
1     [TX]
2    [TXN]
3    [TXN]
4      [T]
Name: Transaction_Ref, dtype: object

##### Question 6: Zero or More (*)
In the Error_Message column, replace the word "attempts:" followed by zero or more spaces  , and then a single digit, with the string "REDACTED".

Hint: Use attempts:\s*\d inside .str.replace(). The \s* handles any number of spaces between the colon and the number.

In [42]:
df['Error_Message'].str.replace(r'attempts:\s*\d',"REDACTED",regex = True)

0          Payment timeout after 30s.
1    Card declined! Error code: 5021.
2          User dropped from session.
3    Retrying connection... REDACTED.
4                            SUCCESS.
Name: Error_Message, dtype: str

##### Question 7: Combining Ranges and Classes
Extract the exact phone numbers that follow the pattern of 3 digits, a hyphen, and then 4 digits.

Hint: Use \d{3}-\d{4} inside .str.findall() on the Phone_Number column.

In [44]:
df['Phone_Number'].str.findall(r'\d{3}-\d{4}')

0            []
1            []
2            []
3    [044-2345]
4    [654-3210]
Name: Phone_Number, dtype: object

##### Question 8: The + with Word Characters
Extract the domain names (e.g., gmail.com, yahoo.com) from the User_Email column.

Hint: Use .str.findall() and look for an @ symbol, followed by one or more word characters \w+, a literal dot \., and then one or more word characters \w+. (Note: You must escape the dot with a backslash).

In [55]:
df['User_Email'].str.findall(r'@\w+.\w+')

0      [@gmail.com]
1      [@yahoo.com]
2        [@zoho.in]
3    [@company.com]
4     [@domain.org]
Name: User_Email, dtype: object

##### Question 9: Open-ended Range ({n,})
Replace any sequence of 2 or more dots . in the Error_Message column (like the ellipsis in row 3) with a single dot ".".

Hint: Use \.{2,} inside .str.replace(). Leaving the second number blank in the curly braces means "or more".

In [53]:
df['Error_Message'].str.replace(r'\.{2,}','.',regex = True)

0           Payment timeout after 30s.
1     Card declined! Error code: 5021.
2           User dropped from session.
3    Retrying connection. attempts: 3.
4                             SUCCESS.
Name: Error_Message, dtype: str

##### Question 10: Putting it Together
Filter the DataFrame to return only rows where the User_Email contains 2 or more consecutive lowercase vowels.

Hint: Combine a custom character class [aeiou] with a quantifier {2,} inside .str.contains().

In [52]:
df['User_Email'].str.contains(r'[aeiou]{2}')

0     True
1     True
2     True
3    False
4     True
Name: User_Email, dtype: bool

##### Here is a brand new dataset themed around an Intrusion Detection System (IDS) log to make the practice a bit more relevant to real-world cybersecurity analysis.

In [1]:
import pandas as pd
import io

csv_data = """Alert_ID,Source_IP,Target_Port,Payload_Signature,Action_Log
ALT-10,192.168.1.50,Port_80,None,Traffic normal. Byte count: 1500.
ALT-20,10.0.2.15,Port_443,SQL_Inject,Blocked input: ' OR 1=1 --
ALT-30,172.16.0.8,Port_22,SSH_Brute,Failed logins: 55. Retrying...
ALT-40,192.168.1.100,Port_8080,SQLi_Attempt,Quarantined file successfully.
ALT-50,10.0.0.5,Port_21,FTP_Anon,Warning:    Dropped 4 packets.
"""

df = pd.read_csv(io.StringIO(csv_data))
df

,Alert_ID,Source_IP,Target_Port,Payload_Signature,Action_Log
0,ALT-10,192.168.1.50,Port_80,NaN,Traffic normal. Byte count: 1500.
1,ALT-20,10.0.2.15,Port_443,SQL_Inject,Blocked input: ' OR 1=1 --
2,ALT-30,172.16.0.8,Port_22,SSH_Brute,Failed logins: 55. Retrying...
3,ALT-40,192.168.1.100,Port_8080,SQLi_Attempt,Quarantined file successfully.
4,ALT-50,10.0.0.5,Port_21,FTP_Anon,Warning: Dropped 4 packets.


1. Extract all full, multi-digit numbers (one or more consecutive digits) from the Action_Log column.

In [2]:
df['Action_Log'].str.findall(r'\d+')

0    [1500]
1    [1, 1]
2      [55]
3        []
4       [4]
Name: Action_Log, dtype: object

2. Extract sequences of exactly 3 digits from the Source_IP column.

In [3]:
df['Source_IP'].str.findall(r'\d{3}')

0         [192, 168]
1                 []
2              [172]
3    [192, 168, 100]
4                 []
Name: Source_IP, dtype: object

3. Filter the DataFrame to return only rows where the Target_Port column contains a number that is strictly between 3 and 4 digits long.

In [4]:
df['Target_Port'].str.contains(r'\d{3,4}')

0    False
1     True
2    False
3     True
4    False
Name: Target_Port, dtype: bool

4. Filter the DataFrame to return only rows where the Payload_Signature column contains the text "SQL", an optional "i", and then an underscore _.

In [12]:
df['Payload_Signature'].str.contains(r'SQL?i_')

0    False
1    False
2    False
3     True
4    False
Name: Payload_Signature, dtype: bool

5. In the Action_Log column, replace the word "Warning:" followed by zero or more spaces, and then the word "Dropped", with the exact string "CRITICAL_DROP".

In [15]:
df['Action_Log'].str.replace(r'Warning:\s*Dropped',"CRITICAL_DROP",regex = True)

0    Traffic normal. Byte count: 1500.
1           Blocked input: ' OR 1=1 --
2       Failed logins: 55. Retrying...
3       Quarantined file successfully.
4             CRITICAL_DROP 4 packets.
Name: Action_Log, dtype: str

6. Replace any sequence of 2 or more literal dots . in the Action_Log column with an empty string "".

In [11]:
df['Action_Log'].str.replace(r'\.{2,}',"",regex = True)

0    Traffic normal. Byte count: 1500.
1           Blocked input: ' OR 1=1 --
2          Failed logins: 55. Retrying
3       Quarantined file successfully.
4       Warning:    Dropped 4 packets.
Name: Action_Log, dtype: str

#### 3. Groups ( ) — The Ultimate Data Engineering Tool

##### In Pandas, Groups are your best friend. While .str.replace() and .str.contains() are great, Groups combined with .str.extract() is how you pull specific data out of a messy log and turn it into a brand-new, clean column.

##### The Golden Rule of Extraction:
Write the Regex pattern for the entire piece of text so Python finds it, but only put parentheses ( ) around the exact part you want to keep.

In [16]:
import pandas as pd
import io

csv_data = """Session_ID,Client_Agent,Request_Route,Auth_Token,System_Note
S-101,Mozilla/5.0 (Windows),/api/v1/login,Token: 98AB-X,User admin logged in from OMR.
S-102,Chrome/91.0 (Mac),/api/v1/dashboard,Token: 44CD-Y,Accessed dashboard at 14:30.
S-103,Python-urllib/3.8,/admin/config,Token: NONE,Failed attempt from T-Nagar!
S-104,PostmanRuntime/7.28,/api/v2/users,Token: 12EF-Z,Fetched 50 users.
S-105,Mozilla/5.0 (Linux),/api/v1/logout,Token: 55GH-W,Session closed for user guest.
"""

df = pd.read_csv(io.StringIO(csv_data))
df

,Session_ID,Client_Agent,Request_Route,Auth_Token,System_Note
0,S-101,Mozilla/5.0 (Windows),/api/v1/login,Token: 98AB-X,User admin logged in from OMR.
1,S-102,Chrome/91.0 (Mac),/api/v1/dashboard,Token: 44CD-Y,Accessed dashboard at 14:30.
2,S-103,Python-urllib/3.8,/admin/config,Token: NONE,Failed attempt from T-Nagar!
3,S-104,PostmanRuntime/7.28,/api/v2/users,Token: 12EF-Z,Fetched 50 users.
4,S-105,Mozilla/5.0 (Linux),/api/v1/logout,Token: 55GH-W,Session closed for user guest.


1. Basic Extraction: Extract the exact alphanumeric token code (e.g., 98AB-X) from the Auth_Token column, ignoring the word "Token: ".

Hint: Match the literal word Token:\s, but only put parentheses around the characters that come after it: (\w{4}-\w)

In [18]:
df['Auth_Token'].str.findall(r'Token:\s(\w{4}-\w)')

0    [98AB-X]
1    [44CD-Y]
2          []
3    [12EF-Z]
4    [55GH-W]
Name: Auth_Token, dtype: object

2. Extracting from the Middle: Extract the API version (e.g., v1 or v2) from the Request_Route column.

Hint: Look for /api/ followed by a v and a digit. Only capture the v and the digit.

In [19]:
df['Request_Route'].str.findall(r'/api/(\w{2})')

0    [v1]
1    [v1]
2      []
3    [v2]
4    [v1]
Name: Request_Route, dtype: object

3. Grouping for Alternation (OR logic): Use .str.contains() to filter the DataFrame and return only rows where the Request_Route is either login or logout.

Hint: You can use the pipe symbol | inside a group to mean "OR", like this: (login|logout).

In [22]:
df['Request_Route'].str.contains(r'/api/\w{2}/(login|logout)')

C:\Users\sakth\AppData\Local\Temp\ipykernel_11812\2075006815.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['Request_Route'].str.contains(r'/api/\w{2}/(login|logout)')


0     True
1    False
2    False
3    False
4     True
Name: Request_Route, dtype: bool

In [23]:
df['Request_Route'].str.contains(r'login|logout')

0     True
1    False
2    False
3    False
4     True
Name: Request_Route, dtype: bool

4. Escaping Literal Parentheses: Extract the Operating System name (e.g., Windows, Mac, Linux) from the Client_Agent column.

Hint: The OS is wrapped in literal parentheses. You must escape them with backslashes \( \), and put your capture group (\w+) between them.

In [37]:
df['Client_Agent'].str.extract(r'\((\w+)\)')

,0
0,Windows
1,Mac
2,NaN
3,NaN
4,Linux


5. Multiple Groups (Two Columns): Extract the 2-digit hour and 2-digit minute from the System_Note column. If you use two capture groups, Pandas will automatically create two separate columns!

Hint: Use (\d{2}):(\d{2}).

In [38]:
df['System_Note'].str.extract(r'(\d{2}):(\d{2})')

,0,1
0,NaN,NaN
1,14,30
2,NaN,NaN
3,NaN,NaN
4,NaN,NaN


6. Prefix and Suffix Split: Extract the Session prefix (the letter S) and the number (e.g., 101) into two separate columns from Session_ID.

Hint: Capture the first letter, match the literal hyphen, and capture the digits.

In [39]:
df['Session_ID'].str.extract(r'(S)-(\d+)')

,0,1
0,S,101
1,S,102
2,S,103
3,S,104
4,S,105


7. Extracting with Context: Extract the specific username that comes immediately after the word "User " or "user " in the System_Note column (e.g., admin, guest).

Hint: Match [Uu]ser\s to find the anchor word, but only capture the word characters (\w+) that follow it.

In [41]:
df['System_Note'].str.extract(r'[Uu]ser\s(\w+)')

,0
0,admin
1,NaN
2,NaN
3,NaN
4,guest


8. Extracting the Browser: Extract the main browser/client name (the word before the forward slash, like Mozilla, Chrome, PostmanRuntime) from the Client_Agent column.

Hint: Capture word characters (\w+), followed immediately by an escaped forward slash \/.

In [42]:
df['Client_Agent'].str.extract(r'(\w+)/')

,0
0,Mozilla
1,Chrome
2,urllib
3,PostmanRuntime
4,Mozilla


9. Targeted Number Extraction: Extract the exact 2-digit number that appears after the forward slash in the Client_Agent column (e.g., 91 from Chrome, or the 28 from Postman).

Hint: Match the slash and any initial numbers, but only capture the specific digits you want.

In [55]:
df['Client_Agent'].str.extract(r'/.*?(\d{2})')

,0
0,NaN
1,91
2,NaN
3,28
4,NaN


10. Complex Multi-Extraction: Extract the first 2 numbers of the Auth Token into one column, and the very last letter of the Auth Token into a second column.



In [54]:
df['Auth_Token'].str.extract(r'Token:\s(\d{2})\w+-(\w)')

,0,1
0,98,X
1,44,Y
2,NaN,NaN
3,12,Z
4,55,W


#### This is the final boss
Since you have been locked in on cybersecurity logic, I have designed this final dataset as a raw Intrusion Detection System (IDS) threat log. It contains highly irregular formatting, nested data, and unpredictable strings.

To solve these, you will need to seamlessly combine your knowledge of Character Classes (\w, \d), Quantifiers (+, {n}, ?), and Groups (). There are no hints.

In [1]:
import pandas as pd
import io

csv_data = """Threat_ID,Source_Node,Payload_Header,Admin_Notes
THRT-9981X,IP:192.168.100.14_MAC:00-14-22-01-23-45,[GET] /v2/api/auth?token=JWT_ABC123,Reviewed by sys_admin. Escalate to tier-2.
THRT-1029Y,IP:10.0.5.55_MAC:A1-B2-C3-D4-E5-F6,[POST] /v1/db/drop?table=users,Automated block applied at 2026-08-31.
WARN-0044Z,IP:172.16.254.1_MAC:11-22-33-44-55-66,[PUT] /admin/config_backup.zip,False positive. Cleared by SecOps-Team!
THRT-5500A,IP:192.168.0.1_MAC:FF-EE-DD-CC-BB-AA,[SSH] root@192.168.0.1:22,Brute force detected. 450 attempts logged.
"""

df = pd.read_csv(io.StringIO(csv_data))
df

,Threat_ID,Source_Node,Payload_Header,Admin_Notes
0,THRT-9981X,IP:192.168.100.14_MAC:00-14-22-01-23-45,[GET] /v2/api/auth?token=JWT_ABC123,Reviewed by sys_admin. Escalate to tier-2.
1,THRT-1029Y,IP:10.0.5.55_MAC:A1-B2-C3-D4-E5-F6,[POST] /v1/db/drop?table=users,Automated block applied at 2026-08-31.
2,WARN-0044Z,IP:172.16.254.1_MAC:11-22-33-44-55-66,[PUT] /admin/config_backup.zip,False positive. Cleared by SecOps-Team!
3,THRT-5500A,IP:192.168.0.1_MAC:FF-EE-DD-CC-BB-AA,[SSH] root@192.168.0.1:22,Brute force detected. 450 attempts logged.


1. Targeted Masking: In the Admin_Notes column, replace any sequence of exactly 3 or more consecutive digits with the string "***".

In [19]:
df['Admin_Notes'].str.replace(r'\d{3,}',"***",regex = True)

0    Reviewed by sys_admin. Escalate to tier-2.
1         Automated block applied at ***-08-31.
2       False positive. Cleared by SecOps-Team!
3    Brute force detected. *** attempts logged.
Name: Admin_Notes, dtype: str

2. Strict Filtering: Filter the DataFrame to return only the rows where the Threat_ID starts with exactly 4 uppercase letters, followed by a hyphen, then exactly 4 digits, and ending with exactly 1 uppercase letter.

In [21]:
df['Threat_ID'].str.contains(r'[A-Z]{4}-\d{4}[A-Z]',regex = True)

0    True
1    True
2    True
3    True
Name: Threat_ID, dtype: bool

3. Complex Extraction (Single Column): Extract the full 17-character MAC address (e.g., 00-14-22-01-23-45) from the Source_Node column into a new column.

In [24]:
df['Source_Node'].str.extract(r'MAC:([\w+-]+)')

,0
0,00-14-22-01-23-45
1,A1-B2-C3-D4-E5-F6
2,11-22-33-44-55-66
3,FF-EE-DD-CC-BB-AA


4. Contextual Extraction: Extract the specific team or user name that immediately follows the exact phrase "by " in the Admin_Notes column (e.g., sys_admin or SecOps-Team). Ensure the word "by " itself is not included in your final output.

In [28]:
df['Admin_Notes'].str.extract(r'by\s(\w+-?\w+)')

,0
0,sys_admin
1,NaN
2,SecOps-Team
3,NaN


5. Multi-Group Extraction (Two Columns): Using a single .str.extract() command on the Payload_Header column, extract the HTTP method/protocol found inside the square brackets (e.g., GET, POST) into the first column, and the exact file path or route that immediately follows the closing bracket and space (e.g., /v2/api/auth) into the second column. Stop extracting before you hit any query parameters (?) or colons (:).

In [55]:
df['Payload_Header'].str.extract(r'\[(\w+)\]\s([^?:]+)')

,0,1
0,GET,/v2/api/auth
1,POST,/v1/db/drop
2,PUT,/admin/config_backup.zip
3,SSH,root@192.168.0.1


In [57]:
import pandas as pd
import io

csv_data = """Log_ID,Timestamp,Endpoint_Target,Security_Payload
L-01,2026-09-01T10:05:12,/api/deepfake/scan?v=2,Status: SUCCESS. Confidence: 98.5%. Trigger: Face_Warp.
L-02,2026-09-01T10:06:00,/api/mobile/auth,Status: FAILED. Intrusion: ARM64_Root. Token: eyJhbGci.
L-03,2026-09-01T10:07:33,/api/deepfake/scan?v=1,Status: SUCCESS. Confidence: 42.1%. Trigger: None.
L-04,2026-09-01T10:08:15,/api/mobile/sys_check,Status: BLOCKED. Intrusion: x86_Emulator. Token: eyJ0eXAi.
"""

df = pd.read_csv(io.StringIO(csv_data))
df

,Log_ID,Timestamp,Endpoint_Target,Security_Payload
0,L-01,2026-09-01T10:05:12,/api/deepfake/scan?v=2,Status: SUCCESS. Confidence: 98.5%. Trigger: F...
1,L-02,2026-09-01T10:06:00,/api/mobile/auth,Status: FAILED. Intrusion: ARM64_Root. Token: ...
2,L-03,2026-09-01T10:07:33,/api/deepfake/scan?v=1,Status: SUCCESS. Confidence: 42.1%. Trigger: N...
3,L-04,2026-09-01T10:08:15,/api/mobile/sys_check,Status: BLOCKED. Intrusion: x86_Emulator. Toke...


##### Question 1: The URL Truncation (Replacement)
In the Endpoint_Target column, completely remove the query parameters. You must find the question mark ? and delete it, along with absolutely every character that comes after it, leaving only the clean base route (e.g., /api/deepfake/scan).

In [95]:
df['Endpoint_Target'].str.replace(r'\?.*',"",regex = True)

0       /api/deepfake/scan
1         /api/mobile/auth
2       /api/deepfake/scan
3    /api/mobile/sys_check
Name: Endpoint_Target, dtype: str

##### Question 2: Highly Specific Filtering (Contains)
Filter the DataFrame to return only the rows where the Security_Payload column logs a deepfake confidence score in the 90s (i.e., any score from 90.0% to 99.9%).

In [102]:
df['Security_Payload'].str.contains(r'Confidence:\s9\d\.\d\%')

0     True
1    False
2    False
3    False
Name: Security_Payload, dtype: bool

##### Question 3: The Ultimate Multi-Extraction (Groups)
Using a single .str.extract() command on the Security_Payload column, extract the exact floating-point Confidence score (e.g., 98.5 or 42.1) into the first column, and the specific Trigger reason (e.g., Face_Warp or None) into the second column. If a row does not contain these values, it should gracefully return NaN for both columns.

In [108]:
df['Security_Payload'].str.extract(r'Confidence:\s(\d{2}.\d)\%\.\sTrigger:\s(\w+)')

,0,1
0,98.5,Face_Warp
1,NaN,NaN
2,42.1,None
3,NaN,NaN
